<a href="https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Author:** Najam ul Hasan · FlyRank ML Internship capstone · October 2026
**Live paper:** https://najam2005.github.io/flyrank-ml-internship/paper/
**Full analysis notebook:** `work/notebooks/capstone_01_decline_prediction.ipynb` (needs the gated Hugging Face data)

This notebook is the short, public-safe version of the paper. It does **not** need the gated data: the code below recomputes the headline numbers from the aggregate tables in the paper and reads the committed receipts (`work/outputs/*.json`, `work/outputs/figs/`). Everything is aggregate; clients are shown as letters.

## 1. Question

*The research question and the decision it supports.*

Can a page's later traffic decline be predicted from its earlier impression history and static page descriptors, on clients the model has never seen? And does any model beat a transparent rule an editor can read?

Decision supported: an SEO editor has time to review only some pages. Which pages should be reviewed first for a possible refresh?

Full write-up: https://najam2005.github.io/flyrank-ml-internship/paper/
Full analysis notebook: work/notebooks/capstone_01_decline_prediction.ipynb

In [1]:
# Setup: libraries and the committed receipts (aggregate files only, no raw data)
import json, urllib.request
import numpy as np
import pandas as pd

RAW = "https://raw.githubusercontent.com/NAJAM2005/flyrank-ml-internship/main/work/outputs/"

def load_json(name):
    try:
        with urllib.request.urlopen(RAW + name, timeout=30) as r:
            return json.load(r)
    except Exception as e:
        print("Could not load", name, "-", e)
        return None

metrics = load_json("capstone_metrics.json")
segments = load_json("capstone_segments.json")
for name, obj in [("capstone_metrics.json", metrics), ("capstone_segments.json", segments)]:
    if isinstance(obj, dict):
        print(name, "top-level keys:", list(obj.keys())[:20])
    elif isinstance(obj, list):
        print(name, "is a list of", len(obj), "items")

capstone_metrics.json top-level keys: ['data', 'lane_comparison', 'key_overlap_with_ai_opportunity', 'mean_client_auc', 'mean_client_auc_excluding_client_B', 'paired_differences_all_clients', 'mean_top20_lift_pp', 'concentration', 'capture_top20_share_of_lost_impressions', 'persistence_check', 'selection_sensitivity_mean_auc', 'review_budget_top20_share_of_lost_impressions']
capstone_segments.json is a list of 12 items


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source.** FlyRank internship lane datasets (pseudonymized June-2026 example cuts, 90-day windows), gated on Hugging Face (`FlyRank/internship-lanes`). No client names, domains, URLs, queries or raw rows appear here.

**Choosing a lane.** A forward-looking label needs two consecutive windows *and* a mix of declining and non-declining pages. I checked all four default lanes:

- `decline_recovery`: 100% of pages had already declined, so it cannot support a prediction.
- `ranking_lifecycle` and `engagement_fix`: one window only.
- `ai_opportunity`: two windows and a mix (46.7% declining). **This is the lane used**, for impression windows only.

**Sample.** Pages need at least 100 impressions in the previous 30 days (11,408 pages). I then kept only pages that also appear in `engagement_fix`, which supplies the static descriptors, leaving **10,550 pages from 29 clients (45.0% declining)**. The 858 pages outside that lane were excluded because they decline more (66.9% vs 45.0%) and lane membership may depend on the label window (a leakage risk). The `engagement_fix` traffic columns were not used: they equal this lane's last window and would leak the label.

**Pre-selected population.** These are pages FlyRank's own rules tagged as AI-opportunity pages, so results describe this subset, not pages in general.

In [2]:
lanes = pd.DataFrame({
    "lane": ["decline_recovery", "ranking_lifecycle", "engagement_fix", "ai_opportunity"],
    "pages": [56253, 205749, 33202, 11571],
    "clients": [46, 54, 44, 33],
    "two_windows": ["yes", "no", "no", "yes"],
    "share_declining": ["100%", "n/a", "n/a", "46.7%"],
})
display(lanes)

# Sample accounting (numbers from the paper)
eligible, kept = 11408, 10550
print("Pages excluded (not in engagement_fix):", eligible - kept)   # expect 858
assert eligible - kept == 858

,lane,pages,clients,two_windows,share_declining
0,decline_recovery,56253,46,yes,100%
1,ranking_lifecycle,205749,54,no,n/a
2,engagement_fix,33202,44,no,n/a
3,ai_opportunity,11571,33,yes,46.7%


Pages excluded (not in engagement_fix): 858


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label.** A page is *declining* (1) if `impressions_last_30d < 0.8 x impressions_prev_30d`, among pages with at least 100 previous-window impressions. The 20% and 100-impression thresholds were fixed before looking at any model results.

**Features (leak-free).** log previous-window impressions, log oldest-window impressions, their difference (momentum), a no-old-traffic flag, page-age tier, and static descriptors (intent, word-count tier, character-count tier). The oldest 30-day window is derived as the 90-day total minus the two later windows, which assumes three exact, non-overlapping periods.

**Left out on purpose.** Every 90-day and last-30-day column (they contain the label window), `trend_*`, `health_score`, `is_declining`, product flags, hashed identifiers (grouping only), and the `engagement_fix` traffic columns.

**Baseline rule (transparent, set before results).** Score = 0.6 x (page age 181+ days) + 0.4 x percentile of previous-window impressions.

**Model.** Gradient boosting (`HistGradientBoostingClassifier`, depth 3, learning rate 0.05, 150 iterations, seed 0).

**Validation.** Leave-one-client-out: train on all other clients, score the held-out one. The headline metric is the mean per-client AUC over the 10 clients with at least 100 pages (a pooled AUC would partly measure which client a page belongs to). Differences use a paired bootstrap over clients (5,000 resamples, seed 0). Because an editor cares about impressions at stake, I also report the share of each client's lost impressions found in its top 20% of pages.

In [3]:
# The label and the baseline rule as code, shown on a TOY example.
# The numbers below are invented, only to show how the two rules work. They are not real data.
def is_declining(prev_30d, last_30d, min_prev=100, drop=0.8):
    """1 if last-30d impressions < 0.8 x previous-30d; None if fewer than 100 previous impressions."""
    if prev_30d < min_prev:
        return None
    return int(last_30d < drop * prev_30d)

def baseline_score(age_days, prev_pct):
    """0.6 x (page age 181+ days) + 0.4 x percentile of previous-window impressions (0 to 1)."""
    return 0.6 * (age_days >= 181) + 0.4 * prev_pct

toy = pd.DataFrame({
    "prev_30d": [1000, 500, 80, 2000],
    "last_30d": [700, 480, 50, 2100],
    "age_days": [200, 120, 400, 60],
})
toy["prev_pct"] = toy["prev_30d"].rank(pct=True)
toy["label"] = [is_declining(p, l) for p, l in zip(toy["prev_30d"], toy["last_30d"])]
toy["baseline_score"] = [baseline_score(a, p) for a, p in zip(toy["age_days"], toy["prev_pct"])]
display(toy)

,prev_30d,last_30d,age_days,prev_pct,label,baseline_score
0,1000,700,200,0.75,1.0,0.9
1,500,480,120,0.50,0.0,0.2
2,80,50,400,0.25,NaN,0.7
3,2000,2100,60,1.00,0.0,0.4


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Leave-one-client-out, 10 evaluated clients. **The model does not reliably beat the rule.** Mean per-client AUC is 0.561 for the age-and-demand rule and 0.567 for boosting; the paired difference is +0.006 (95% interval -0.046 to +0.052). The main model beats the rule in 6 of 10 clients, and gains and losses go in both directions.

On the editor's own objective (share of a client's lost impressions found in its top 20% of pages), the classifier finds 20.8% (random is 20.0%), the baseline rule 50.9%, and simply ranking by page impressions 56.1%. Multiplying model risk by impressions adds nothing (-0.5 points). Part of the 56% is a size effect: a 20% drop on a large page is more impressions than on a small one.

The cells below recompute the AUC comparison from the per-client table in the paper (rounded to 3 decimals, so the bootstrap interval may differ slightly from the paper's in the last digit).

In [4]:
clients = pd.DataFrame({
    "client":       list("ABCDEFGHIJ"),
    "pages":        [3925, 3452, 938, 682, 457, 314, 146, 132, 109, 100],
    "decline_rate": [0.445, 0.567, 0.459, 0.177, 0.411, 0.172, 0.240, 0.470, 0.239, 0.170],
    "baseline_auc": [0.514, 0.601, 0.510, 0.656, 0.471, 0.576, 0.632, 0.563, 0.573, 0.510],
    "model_auc":    [0.507, 0.614, 0.578, 0.485, 0.513, 0.607, 0.569, 0.612, 0.534, 0.649],
})
display(clients)

print("Mean per-client AUC  baseline:", round(clients["baseline_auc"].mean(), 3),
      "| model:", round(clients["model_auc"].mean(), 3))
print("Clients above 0.5    baseline:", int((clients["baseline_auc"] > 0.5).sum()), "/ 10",
      "| model:", int((clients["model_auc"] > 0.5).sum()), "/ 10")
print("Clients where model beats baseline:", int((clients["model_auc"] > clients["baseline_auc"]).sum()), "/ 10")

# Paired bootstrap over clients (5,000 resamples, seed 0)
diff = (clients["model_auc"] - clients["baseline_auc"]).to_numpy()
rng = np.random.default_rng(0)
boot = np.array([rng.choice(diff, size=len(diff), replace=True).mean() for _ in range(5000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Mean difference (model - baseline): {diff.mean():+.3f}  95% interval ({lo:+.3f}, {hi:+.3f})")
print("Interval includes zero ->", bool(lo <= 0 <= hi))

,client,pages,decline_rate,baseline_auc,model_auc
0,A,3925,0.445,0.514,0.507
1,B,3452,0.567,0.601,0.614
2,C,938,0.459,0.510,0.578
3,D,682,0.177,0.656,0.485
4,E,457,0.411,0.471,0.513
5,F,314,0.172,0.576,0.607
6,G,146,0.240,0.632,0.569
7,H,132,0.470,0.563,0.612
8,I,109,0.239,0.573,0.534
9,J,100,0.170,0.510,0.649


Mean per-client AUC  baseline: 0.561 | model: 0.567
Clients above 0.5    baseline: 9 / 10 | model: 9 / 10
Clients where model beats baseline: 6 / 10
Mean difference (model - baseline): +0.006  95% interval (-0.046, +0.052)
Interval includes zero -> True


In [5]:
# The editor's objective: share of lost impressions found in the top 20% of each client's pages
# (mean over the 10 evaluated clients; values from the paper / capstone_metrics.json)
capture = pd.DataFrame({
    "ranking": ["Random", "Boosting (main model)", "Baseline rule (age + demand)",
                "Expected loss (model risk x impressions)", "Impressions only (demand)",
                "Oracle (knows true losses)"],
    "share_of_lost_impressions_in_top20pct": [20.0, 20.8, 50.9, 55.6, 56.1, 90.0],
})
display(capture)

,ranking,share_of_lost_impressions_in_top20pct
0,Random,20.0
1,Boosting (main model),20.8
2,Baseline rule (age + demand),50.9
3,Expected loss (model risk x impressions),55.6
4,Impressions only (demand),56.1
5,Oracle (knows true losses),90.0


## 5. Limitations

*What this work cannot claim.*

1. **The model does not beat a simple rule**, and is the wrong tool for the editor's goal (it finds 20.8% of lost impressions in the top 20% of pages; random is 20.0%).
2. **Weak where the value is.** One client holds 37.2% of pages and 46.9% of lost impressions, and its AUC is about 0.51 (close to random).
3. **One pair of windows.** Leave-one-client-out tests transfer across clients, not across time. Everything comes from one June-2026 cut, so seasonality or an algorithm update would hit every client at once.
4. **Pre-selected population and sample selection** (an AI-opportunity subset; 858 pages dropped).
5. **Regression to the mean and a broad decline.** 45% of pages are labelled declining, so part of the label is noise or a market-wide dip, not page-specific decay.
6. **Small evaluation set.** Only 10 of 29 clients have at least 100 pages; the bootstrap has little power.
7. **Concentrated losses.** The top 1% of pages hold 19.8% of lost impressions and the top 10% hold 65.6%.
8. **Segments are in-sample**, defined after seeing the data; some rest on 3 or 4 clients.
9. **Coarse, possibly late features** (age and word counts are tiers; the oldest-window feature rests on a window-alignment assumption).
10. **Impressions only.** A fall can reflect reporting changes, seasonality or demand, not a content problem.
11. **Observational only.** No claim that refreshing a page changes its traffic, and nothing about how Google ranks pages.
12. **Scores are not calibrated per client.** They are risk scores for ranking within a client, not probabilities.

Two development checks (old-window sanity checks and the leak ablation) were run on an earlier 11,408-page sample and are not in the final notebook; the paper says so.

In [6]:
# Concentration check, computed from the per-client table above
a = clients.loc[clients["client"] == "A"].iloc[0]
print("Client A share of pages:", round(100 * a["pages"] / 10550, 1), "%  (paper: 37.2%)")
print("Client A model AUC:", a["model_auc"], "(close to 0.5 = random)")
print("Clients evaluated:", len(clients), "of 29 (those with at least 100 pages)")

Client A share of pages: 37.2 %  (paper: 37.2%)
Client A model AUC: 0.507 (close to 0.5 = random)
Clients evaluated: 10 of 29 (those with at least 100 pages)


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

A decision-support playbook, not a causal claim. Within each client, rank pages by impressions at risk (previous-window impressions), apply the age and demand rules below, and read the top pages by hand. **Confidence: low to moderate** for the ordering; low for any claim that a score identifies which individual page will fall.

In the page-level action list, a top-20%-per-client review budget (2,098 pages) held 56.4% of all lost impressions. REVIEW_FIRST pages (2,778) decline at 49.0% vs a 45.0% base rate but hold 57.3% of lost impressions, mostly because of their size. LOW_PRIORITY pages (779) decline at 24.9% and hold 2.7%. All figures are in-sample and directional.

In [7]:
recs = pd.DataFrame([
    [1, "181-365 days, high demand", "54.7% decline (+9.6 pts vs base); 24.9% of lost impressions (29.0% without top client); above own rate in 3/4 clients", "REVIEW_FIRST (thin client evidence)"],
    [2, "91-180 days, high demand", "44.4% decline but 32.4% of lost impressions (36.7% without top client); above own rate in 7/9", "REVIEW_FIRST (for volume at stake)"],
    [3, "181-365 days, low demand", "53.3% decline (+8.3 pts) but only 2.4% of loss", "BATCH_REVIEW"],
    [4, "365+ days, high demand", "14.4% of loss falls to 6.7% without the top client; 38.5% decline", "MONITOR (client-dependent)"],
    [5, "Other mid-age and older segments", "Mixed or small", "MONITOR"],
    [6, "31-90 days, any demand", "17-24 points below base; below own rate in 0/3, 0/4, 0/5 clients", "LOW_PRIORITY"],
], columns=["priority", "segment", "evidence", "action_label"])
pd.set_option("display.max_colwidth", None)
display(recs)

,priority,segment,evidence,action_label
0,1,"181-365 days, high demand",54.7% decline (+9.6 pts vs base); 24.9% of lost impressions (29.0% without top client); above own rate in 3/4 clients,REVIEW_FIRST (thin client evidence)
1,2,"91-180 days, high demand",44.4% decline but 32.4% of lost impressions (36.7% without top client); above own rate in 7/9,REVIEW_FIRST (for volume at stake)
2,3,"181-365 days, low demand",53.3% decline (+8.3 pts) but only 2.4% of loss,BATCH_REVIEW
3,4,"365+ days, high demand",14.4% of loss falls to 6.7% without the top client; 38.5% decline,MONITOR (client-dependent)
4,5,Other mid-age and older segments,Mixed or small,MONITOR
5,6,"31-90 days, any demand","17-24 points below base; below own rate in 0/3, 0/4, 0/5 clients",LOW_PRIORITY


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper embeds three figures (per-client AUC, capture curves, share of lost impressions by segment) and the tables above. The figure files are committed in `work/outputs/figs/`; the cell below lists and displays them from the repo. The receipts `capstone_metrics.json` (every headline number) and `capstone_segments.json` (pooled-tier segment table) are in `work/outputs/`.

Live paper: https://najam2005.github.io/flyrank-ml-internship/paper/

In [8]:
from IPython.display import Image, display

api = "https://api.github.com/repos/NAJAM2005/flyrank-ml-internship/contents/work/outputs/figs"
try:
    with urllib.request.urlopen(api, timeout=30) as r:
        files = [f for f in json.load(r) if f["name"].lower().endswith(".png")]
    for f in files:
        print(f["name"])
        display(Image(url=f["download_url"], width=700))
except Exception as e:
    print("Could not list figures:", e)

capture_curve.png


loss_by_segment.png


loss_by_segment_within_client.png


per_client_auc.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

## ML-12: communication pack

### 5-minute demo outline

| Time | Beat | What I say / show |
|---|---|---|
| 0:00-0:30 | The decision | Editors can review only some pages. Which should they look at first when traffic is slipping? |
| 0:30-1:30 | Data and lane choice | I checked four lanes. Only `ai_opportunity` had two windows and a mix of declining and non-declining pages. 10,550 pages, 29 clients, 45.0% declining. |
| 1:30-2:30 | Method | Label: last-30-day impressions below 0.8 x previous 30 days. Leak-free features. Leave-one-client-out, so every test client is unseen. A transparent age-and-demand rule as the baseline. |
| 2:30-3:30 | Results | Mean per-client AUC 0.561 (rule) vs 0.567 (boosting); the difference is not reliable. On lost impressions in the top 20% of pages: classifier 20.8%, rule 50.9%, impressions only 56.1%. |
| 3:30-4:30 | What an editor does | Rank pages by impressions at risk within each client; push down pages aged 31-90 days; review the largest 91-365 day pages first; treat reason codes as prompts, not proof. |
| 4:30-5:00 | Honest limits and next step | One time snapshot, a pre-selected population, one client dominating the losses. Next: test across time windows. |

### Social-post cut

I finished my FlyRank ML internship capstone. The question: can we predict which pages will lose traffic, on clients the model has never seen?

What I found: a simple age-and-demand rule (AUC 0.561) did about as well as gradient boosting (0.567). Ranking pages by impressions found far more of the lost traffic than the classifier did (56.1% vs 20.8% in the top 20% of pages). So I shipped a transparent review list instead of a black box.

Observed and directional, not causal. Paper and code: https://najam2005.github.io/flyrank-ml-internship/paper/

Built on the FlyRank internship dataset (flyrank.ai). #MachineLearning #SEO #DataScience

### Employer-facing summary (3 sentences)

I built a leak-aware, leave-one-client-out study on pseudonymized FlyRank page data (10,550 pages, 29 clients) to test whether page-level traffic decline can be predicted and whether a model beats a transparent rule. The model did not reliably beat the rule (mean per-client AUC 0.567 vs 0.561), so I delivered a transparent, impression-ranked review list with reason codes and clearly stated limits. The project shows I can frame a decision, guard against data leakage, test honestly against a baseline, and report a null result plainly.